# DESCRIPTIVE ANALYSIS WORK STATION

**SUBTITLE**: DATA CLEANING, INTERPRETATION, AND ANALYSIS

**AUTHORS**:

- name: Kay-Lee Dramat
- name: Ghofran Mahmoud

# DATA CLEANING, TRANSFORMATION, INTERPRETATION, AND ANALYSIS

WRITE A SHORT DESCRIPTION OF WHAT WILL BE DONE UNDER THIS SECTION. THIS CAN BE A GENERAL DESCRIPTION OF THE WORK STATION OR A DETAILED DESCRIPTION OF THE STEPS YOU WILL TAKE TO CLEAN, TRANSFORM, INTERPRET, AND ANALYZE THE DATA. YOU CAN ALSO USE THIS SPACE TO DESCRIBE THE STRUCTURE OF THIS WORK STATION AND HOW IT CONNECTS TO THE OTHER PAGES IN YOUR PROJECT.

# TOOLS AND LIBRARIES

*AN IMPORTANT NOTE: BEFORE RUNNING THE CODE IN THIS WORK STATION, MAKE SURE TO INSTALL THE NECESSARY LIBRARIES AND TOOLS FOR DATA CLEANING, TRANSFORMATION, INTERPRETATION, AND ANALYSIS. THIS MAY INCLUDE PANDAS, NUMPY, MATPLOTLIB, SEABORN, OR ANY OTHER LIBRARY YOU PLAN TO USE FOR YOUR ANALYSIS.*

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.ticker as mticker
import seaborn as sns
import ast
import re
import warnings
from scipy import stats

warnings.filterwarnings('ignore')

# Global plot style 
DARK_BG   = '#0d0d1a'
PANEL_BG  = '#16162a'
GRID_COL  = '#2a2a45'
TEXT_COL  = '#dde0f0'
MUTED_COL = '#8888aa'
GOLD      = '#f5a623'
CYAN      = '#00c9e0'
CORAL     = '#ff6b6b'
PURPLE    = '#a29bfe'
GREEN     = '#55efc4'
PINK      = '#fd79a8'
BLUE      = '#74b9ff'
PALETTE   = [GOLD, CYAN, CORAL, PURPLE, GREEN, PINK, BLUE]

plt.rcParams.update({
    'figure.facecolor' : DARK_BG,
    'axes.facecolor'   : PANEL_BG,
    'axes.edgecolor'   : GRID_COL,
    'axes.labelcolor'  : TEXT_COL,
    'axes.titlesize'   : 13,
    'axes.titleweight' : 'bold',
    'axes.labelsize'   : 11,
    'xtick.color'      : MUTED_COL,
    'ytick.color'      : MUTED_COL,
    'text.color'       : TEXT_COL,
    'grid.color'       : GRID_COL,
    'grid.linestyle'   : '--',
    'grid.alpha'       : 0.6,
    'legend.facecolor' : '#1e1e35',
    'legend.edgecolor' : GRID_COL,
    'legend.fontsize'  : 9,
    'font.family'      : 'DejaVu Sans',
    'figure.dpi'       : 130,
})

# Shared helper functions 

EXCHANGE_RATES = {
    'USD': 1.0,   'EUR': 1.08,  'GBP': 1.27,  'CAD': 0.74,  'AUD': 0.65,
    'JPY': 0.0067,'INR': 0.012, 'KRW': 0.00073,'CNY': 0.14, 'HKD': 0.128,
    'SGD': 0.75,  'NZD': 0.61,  'BRL': 0.20,   'MXN': 0.058, 'RUB': 0.011,
    'TRY': 0.031, 'PLN': 0.25,  'CHF': 1.13,   'ZAR': 0.055, 'DEM': 0.55,
    'FRF': 0.165, 'ITL': 0.00056,'NLG': 0.49,  'BEF': 0.027, 'ESP': 0.0065,
    'SEK': 0.096, 'NOK': 0.093, 'DKK': 0.145,  'ILS': 0.27,  'TWD': 0.031,
}

SYMBOL_MAP = [
    ('CA$','CAD'),('A$','AUD'),('HK$','HKD'),('NZ$','NZD'),('MX$','MXN'),
    ('NT$','TWD'),('R$','BRL'),('SGD','SGD'),('SEK','SEK'),('NOK','NOK'),
    ('DKK','DKK'),('DEM','DEM'),('FRF','FRF'),('ITL','ITL'),('BEF','BEF'),
    ('ESP','ESP'),('NLG','NLG'),('RUB','RUB'),('RUR','RUB'),('TRY','TRY'),
    ('TRL','TRY'),('PLN','PLN'),('CHF','CHF'),('ZAR','ZAR'),('ILS','ILS'),
    ('$','USD'),
]

def detect_currency(text):
    for pattern, code in SYMBOL_MAP:
        if pattern in text:
            return code
    return 'USD'

def parse_money(value):
    if pd.isna(value) or str(value).strip() == '':
        return None
    s = str(value).strip()
    rate = EXCHANGE_RATES.get(detect_currency(s), 1.0)
    digits = re.sub(r'[^\d.]', '', s)
    if not digits:
        return None
    try:
        return round(float(digits) * rate, 2)
    except ValueError:
        return None

def parse_list_col(value):
    if pd.isna(value):
        return []
    try:
        result = ast.literal_eval(value)
        return result if isinstance(result, list) else []
    except (ValueError, SyntaxError):
        return re.findall(r"'([^']+)'", str(value))

def parse_votes(value):
    if pd.isna(value):
        return None
    s = str(value).strip().replace(',', '')
    mult = 1
    if s.upper().endswith('K'):
        mult, s = 1_000, s[:-1]
    elif s.upper().endswith('M'):
        mult, s = 1_000_000, s[:-1]
    try:
        return int(float(s) * mult)
    except ValueError:
        return None

def fmt_M(x, _):
    return f'${x/1e9:.1f}B' if abs(x) >= 1e9 else f'${x/1e6:.0f}M'

def significance_stars(p):
    if p < 0.001: return '***'
    if p < 0.01:  return '**'
    if p < 0.05:  return '*'
    return 'ns'

print("Libraries and helpers loaded successfully.")

In [ ]:


import pandas as pd

df = pd.read_csv("Data/final_dataset.csv")

print(df.shape)
df.head()




In [ ]:


print(df.columns.tolist())




In [ ]:


import numpy as np
import re
import ast

def parse_duration(duration_str):
    if pd.isna(duration_str):
        return np.nan
    
    duration_str = str(duration_str)
    
    hours = 0
    minutes = 0
    
    h = re.search(r'(\d+)h', duration_str)
    m = re.search(r'(\d+)m', duration_str)
    
    if h:
        hours = int(h.group(1))
    if m:
        minutes = int(m.group(1))
    
    return hours * 60 + minutes

def parse_votes(vote_str):
    if pd.isna(vote_str):
        return np.nan
    
    vote_str = str(vote_str).replace(",", "")
    
    if "K" in vote_str:
        return float(vote_str.replace("K", "")) * 1000
    if "M" in vote_str:
        return float(vote_str.replace("M", "")) * 1000000
    
    try:
        return float(vote_str)
    except:
        return np.nan

def parse_list(x):
    try:
        return ast.literal_eval(x)
    except:
        return []

df["duration_min"] = df["duration"].apply(parse_duration)
df["votes_num"] = df["votes"].apply(parse_votes)
df["genres_list"] = df["genres"].apply(parse_list)
df["countries_list"] = df["countries_origin"].apply(parse_list)

df[["duration", "duration_min", "votes", "votes_num"]].head()




In [ ]:


country_df = df.copy()

# split multiple countries into separate rows
country_df = country_df.explode("countries_list")

# rename for clarity
country_df = country_df.rename(columns={"countries_list": "country"})

# clean country column
country_df["country"] = country_df["country"].astype(str).str.strip()

# remove empty values
country_df = country_df[(country_df["country"] != "") & (country_df["country"] != "[]")]

# group and calculate average rating
country_summary = (
    country_df.groupby("country")
    .agg(
        n_movies=("id", "count"),
        avg_rating=("rating", "mean")
    )
    .reset_index()
)

# keep only countries with enough data
country_summary = country_summary[country_summary["n_movies"] >= 50]

# sort by rating
country_summary = country_summary.sort_values("avg_rating", ascending=False)

country_summary.head(15)





Research Question 1¶How do movie ratings vary across countries of origin?
Interpretation¶This plot shows the top 15 countries with the highest average movie ratings.
We observe that countries such as Iran and Czechoslovakia rank highly.
This may suggest that smaller film industries produce fewer but more critically acclaimed films.
However, this could also reflect selection bias, where only the highest-quality films gain international visibility.


In [ ]:


import matplotlib.pyplot as plt
import numpy as np

top_countries = country_summary.head(15)

# create gradient colors
colors = plt.cm.viridis(np.linspace(0, 1, len(top_countries)))

plt.figure(figsize=(10, 6))
plt.barh(top_countries["country"], top_countries["avg_rating"], color=colors)

plt.xlabel("Average Rating")
plt.ylabel("Country")
plt.title("Top 15 Countries by Average Movie Rating")

plt.gca().invert_yaxis()
plt.tight_layout()
plt.show()





Research Question 2¶Which genres receive the highest ratings and most audience engagement (votes)?


In [ ]:


genre_df = df.copy()

# split genres into separate rows
genre_df = genre_df.explode("genres_list")

# rename
genre_df = genre_df.rename(columns={"genres_list": "genre"})

# clean
genre_df["genre"] = genre_df["genre"].astype(str).str.strip()
genre_df = genre_df[(genre_df["genre"] != "") & (genre_df["genre"] != "[]")]

# group
genre_summary = (
    genre_df.groupby("genre")
    .agg(
        n_movies=("id", "count"),
        avg_rating=("rating", "mean"),
        avg_votes=("votes_num", "mean")
    )
    .reset_index()
)

# filter for reliability
genre_summary = genre_summary[genre_summary["n_movies"] >= 50]

# sort by rating
genre_summary = genre_summary.sort_values("avg_rating", ascending=False)

genre_summary.head(15)




In [ ]:


import numpy as np
import matplotlib.pyplot as plt

top_genres = genre_summary.head(15)

colors = plt.cm.plasma(np.linspace(0, 1, len(top_genres)))

plt.figure(figsize=(10, 6))
plt.barh(top_genres["genre"], top_genres["avg_rating"], color=colors)

plt.xlabel("Average Rating")
plt.ylabel("Genre")
plt.title("Top 15 Genres by Average Movie Rating")

plt.gca().invert_yaxis()
plt.tight_layout()
plt.show()





Interpretation¶This plot shows the top 15 genres with the highest average movie ratings.
We observe that certain genres consistently receive higher ratings, indicating stronger audience appreciation.
Genres with more niche or artistic appeal may rank higher in ratings, while more mainstream genres may attract broader audiences but not always the highest scores.
This highlights a distinction between popularity (votes) and perceived quality (ratings).


In [ ]:


top_votes = genre_summary.sort_values("avg_votes", ascending=False).head(15)

colors = plt.cm.viridis(np.linspace(0, 1, len(top_votes)))

plt.figure(figsize=(10, 6))
plt.barh(top_votes["genre"], top_votes["avg_votes"], color=colors)

plt.xlabel("Average Votes")
plt.ylabel("Genre")
plt.title("Top 15 Genres by Audience Engagement (Votes)")

plt.gca().invert_yaxis()
plt.tight_layout()
plt.show()





Interpretation (Votes)¶This plot highlights genres that receive the highest audience engagement.
We may observe that more mainstream or widely distributed genres attract significantly more votes, even if they do not always have the highest ratings.
This suggests a difference between popularity and critical evaluation.



Research Question 3¶How does movie duration influence audience ratings and votes?


In [ ]:


duration_df = df.copy()

duration_df = duration_df.dropna(subset=["duration_min", "rating", "votes_num"])

duration_df.head()




In [ ]:


plt.figure(figsize=(10, 6))
plt.scatter(duration_df["duration_min"], duration_df["rating"], alpha=0.3)

plt.xlabel("Duration (minutes)")
plt.ylabel("Rating")
plt.title("Movie Duration vs Rating")

plt.tight_layout()
plt.show()





Interpretation¶This plot explores the relationship between movie duration and audience ratings.
We observe that most movies fall within a typical duration range of approximately 80 to 150 minutes. Within this range, ratings are widely distributed, indicating that duration alone does not determine how highly a movie is rated.
There is no clear linear relationship between duration and ratings, suggesting that longer movies are not necessarily better or worse in terms of audience evaluation.
This implies that other factors such as genre, storytelling quality, and production value likely play a more significant role in determining a movie’s success.


In [ ]:


plt.figure(figsize=(10, 6))
plt.scatter(duration_df["duration_min"], duration_df["votes_num"], alpha=0.3)

plt.xlabel("Duration (minutes)")
plt.ylabel("Votes")
plt.title("Movie Duration vs Audience Engagement")

plt.tight_layout()
plt.show()





Interpretation (Votes)¶This plot examines how movie duration relates to audience engagement, measured by the number of votes.
We observe that movies of varying durations can achieve high engagement, although extremely short or very long movies appear less frequently.
There is no strong pattern indicating that longer movies consistently receive more votes. This suggests that audience engagement is influenced more by popularity, accessibility, and genre rather than duration alone.


## Q4: How do box office revenues differ across industries?

This section investigates whether different movie genres consistently earn more or less money at the box office, and whether that pattern holds across all three revenue measures; opening weekend gross, total worldwide gross, and US/Canada gross. Understanding these differences helps studios and producers make informed decisions about which genres to invest in and how to set revenue expectations.

### Data Cleaning

This step prepares the raw revenue and genre columns for analysis. Revenue values stored as formatted text strings in multiple currencies are converted to numeric USD figures. Genre lists stored as text are parsed to extract the primary genre. Rows with no genre label or with no revenue data at all are removed, and genres represented by fewer than 10 movies are excluded to ensure group averages are reliable.

In [ ]:
# =============================================================================
# Q4 — DATA CLEANING
# Target columns: genres, opening_weekend_gross, gross_worldwide, gross_us_canada
# =============================================================================
INPUT_FILE = "data/final_dataset.csv"   
df_raw = pd.read_csv(INPUT_FILE)

print(f"Dataset loaded: {df_raw.shape[0]:,} rows × {df_raw.shape[1]} columns")
print(f"Columns: {list(df_raw.columns)}")

q4_raw = df_raw[['id', 'title', 'genres',
                  'opening_weekend_gross',
                  'gross_worldwide',
                  'gross_us_canada']].copy()

print(f"Starting rows: {len(q4_raw):,}")
print(f"\nMissing values before cleaning:")
print(q4_raw.isnull().sum().to_string())

# Step 1: Parse genre lists and extract the primary (first) genre
q4_raw['genres_list']   = q4_raw['genres'].apply(parse_list_col)
q4_raw['primary_genre'] = q4_raw['genres_list'].apply(lambda x: x[0] if x else None)

# Step 2: Convert money strings to numeric USD
for col in ['opening_weekend_gross', 'gross_worldwide', 'gross_us_canada']:
    q4_raw[col + '_num'] = q4_raw[col].apply(parse_money)

# Step 3: Filter: keep rows with a known genre AND at least one revenue figure
q4 = q4_raw[
    q4_raw['primary_genre'].notna() &
    (
        q4_raw['opening_weekend_gross_num'].notna() |
        q4_raw['gross_worldwide_num'].notna()        |
        q4_raw['gross_us_canada_num'].notna()
    )
].copy()

# Step 4: Drop genres with fewer than 10 movies that have revenue data
genre_counts = q4['primary_genre'].value_counts()
valid_genres  = genre_counts[genre_counts >= 10].index
q4 = q4[q4['primary_genre'].isin(valid_genres)].copy()

print(f"\nRows after cleaning: {len(q4):,}")
print(f"Valid genres (\u226510 movies with revenue): {q4['primary_genre'].nunique()}")
print(f"\nMissing values after cleaning:")
print(q4[['opening_weekend_gross_num','gross_worldwide_num','gross_us_canada_num']].isnull().sum().to_string())

The raw dataset stores revenue figures as formatted text strings (e.g., `"$36,389,705"` or `"€150,000 (estimated)"`) across three columns: `opening_weekend_gross`, `gross_worldwide`, and `gross_us_canada`. These must be parsed and converted to numeric USD before any comparison can be made. The genre column is stored as a stringified Python list, so it is parsed with `ast.literal_eval` and the first item is used as the primary genre. Rows with no genre label and rows belonging to genres with fewer than 10 revenue observations are dropped, as they are too sparse to produce meaningful group averages.

### Data Transformation: Creating X Input and Y Output Variables

The genre column becomes the grouping variable (X input). The three revenue columns — opening weekend, worldwide, and US/Canada gross — are the outcome variables (Y outputs). A fourth engineered variable, international gross (worldwide minus US/Canada), is added to measure how much each genre earns outside North America. A genre-level summary table is then computed showing the mean, median, and breakdown of each revenue stream per genre.

In [ ]:
# =============================================================================
# Q4 — DATA TRANSFORMATION
# =============================================================================

# Select and rename the columns needed for analysis
q4 = q4[[
    'id', 'title', 'primary_genre',
    'opening_weekend_gross_num',
    'gross_worldwide_num',
    'gross_us_canada_num'
]].copy()

q4.columns = [
    'id', 'title', 'genre',
    'opening_weekend', 'worldwide', 'us_canada'
]

# ENGINEERED VARIABLE: International Gross = Worldwide revenue minus US/Canada revenue
q4['international'] = q4['worldwide'] - q4['us_canada']

# SUMMARY TABLE: genre-level descriptive statistics
q4_summary = (
    q4.groupby('genre')
    .agg(
        n_movies          = ('title',        'count'),
        avg_worldwide     = ('worldwide',     'mean'),
        median_worldwide  = ('worldwide',     'median'),
        avg_opening       = ('opening_weekend','mean'),
        median_opening    = ('opening_weekend','median'),
        avg_us_canada     = ('us_canada',     'mean'),
        avg_international = ('international', 'mean'),
    )
    .reset_index()
    .sort_values('avg_worldwide', ascending=False)
)

disp = q4_summary.head(12).copy()
for col in ['avg_worldwide','median_worldwide','avg_opening',
            'median_opening','avg_us_canada','avg_international']:
    disp[col] = disp[col].apply(lambda x: f'${x/1e6:.1f}M' if pd.notna(x) else 'N/A')

print("Summary Table — Average Revenue by Genre (top 12 by worldwide gross):\n")
print(disp.to_string(index=False))

In [ ]:
# =============================================================================
# EXPORT Q4 FINAL DATASET
# =============================================================================
# Create 'outputs' folder 
import os
os.makedirs("outputs", exist_ok=True)
# EXPORT Q4 FINAL DATASET
q4.to_csv("outputs/q4_genre_revenue_dataset.csv", index=False)
q4_summary.to_csv("outputs/q4_genre_summary.csv", index=False)

The transformation step produces four numeric revenue variables per movie: `opening_weekend`, `worldwide`, `us_canada`, and the engineered variable `international` (worldwide minus US/Canada). The genre column serves as the grouping variable (X input), while the four revenue figures are the outcome variables (Y outputs). A genre-level summary table is computed using both mean and median — the mean captures total economic weight, while the median is more robust to the blockbuster outliers that are common in film revenue data.

### Data visualization

In [ ]:
# Figure Q4-1: Average worldwide gross revenue by genre. Only genres with at least 10 movies with revenue data are shown.
# Bars are sorted by average worldwide gross in descending order.

# PLOT Q4-1: Horizontal bar: Average worldwide gross by genre (top 15) 
top15 = q4_summary.head(15).copy()
colors = [PALETTE[i % len(PALETTE)] for i in range(len(top15))]

fig, ax = plt.subplots(figsize=(13, 7))
bars = ax.barh(
    top15['genre'][::-1],
    top15['avg_worldwide'][::-1] / 1e6,
    color=colors[::-1], edgecolor='none', height=0.65
)
for bar in bars:
    w = bar.get_width()
    ax.text(w + 0.5, bar.get_y() + bar.get_height() / 2,
            f'${w:.0f}M', va='center', ha='left', fontsize=8, color=MUTED_COL)

ax.set_xlabel('Average Worldwide Gross (USD Millions)')
ax.set_title('Q4 — Average Worldwide Gross Revenue by Genre\n'
             '(genres with \u2265 10 movies with revenue data, sorted by mean)')
ax.xaxis.set_major_formatter(mticker.FuncFormatter(lambda x, _: f'${x:.0f}M'))
ax.set_xlim(0, top15['avg_worldwide'].max() / 1e6 * 1.2)
ax.grid(axis='x'); ax.set_axisbelow(True)
ax.spines[['top','right','bottom']].set_visible(False)
fig.tight_layout()
plt.show()

Action Epic ($363M), Dinosaur Adventure ($349M), and Superhero ($310M) dominate box office revenue, driven by franchise properties, large budgets, and wide global releases. The steep drop to narrative genres like Drama ($5.7M) and Documentary ($0.8M) confirms that genre is a strong predictor of commercial performance.

In [ ]:
# Figure Q4-2: Revenue stream breakdown for the top 10 genres. Each bar is divided into three segments:
# opening weekend earnings, additional domestic (US/Canada) revenue, and international revenue.
# This breakdown reveals whether a genre's success is driven by its launch, its domestic long-tail,
# or its global appeal.

# PLOT Q4-2: Stacked bar — Revenue stream breakdown (top 10 genres) 
top10 = q4_summary.head(10).set_index('genre').copy()
top10['us_ca_excl_opening'] = (top10['avg_us_canada'] - top10['avg_opening']).clip(lower=0)
top10['intl']               = top10['avg_international'].fillna(0)
top10['opening']            = top10['avg_opening'].fillna(0)

streams       = top10[['opening', 'us_ca_excl_opening', 'intl']] / 1e6
stream_labels = ['Opening Weekend', 'Additional US/Canada', 'International']
stream_colors = [GOLD, CYAN, CORAL]

fig, ax = plt.subplots(figsize=(13, 6))
bottom = np.zeros(len(streams))
for col, label, color in zip(streams.columns, stream_labels, stream_colors):
    vals = streams[col].values
    ax.bar(streams.index, vals, bottom=bottom, label=label,
           color=color, edgecolor='none', alpha=0.88, width=0.6)
    for i, (v, b) in enumerate(zip(vals, bottom)):
        if v > 5:
            ax.text(i, b + v / 2, f'${v:.0f}M',
                    ha='center', va='center', fontsize=7.5,
                    color='white', fontweight='bold')
    bottom += vals

ax.set_ylabel('Average Revenue (USD Millions)')
ax.set_title('Q4 — Revenue Stream Breakdown by Genre — Top 10\n'
             '(Opening Weekend  |  Additional US/Canada  |  International)')
ax.yaxis.set_major_formatter(mticker.FuncFormatter(lambda x, _: f'${x:.0f}M'))
ax.legend(loc='upper right')
ax.set_axisbelow(True)
ax.spines[['top','right']].set_visible(False)
plt.xticks(rotation=30, ha='right')
fig.tight_layout()
plt.show()

International revenue dominates for most top genres; Action Epic earns $241M internationally versus $144M domestically, reflecting strong global demand for visually universal content. Opening weekend contributes only ~11% of total earnings for Action Epic, indicating these films sustain long theatrical runs rather than relying solely on launch day.

In [ ]:
# Figure Q4-3: Box plot of worldwide gross distribution for the top 8 genres.
# The white horizontal line represents the median. The coloured box spans the interquartile range (IQR).
# Dots beyond the whiskers are outlier movies. A wide box and many high outliers indicate a genre where
# a few blockbusters earn dramatically more than the typical movie.

# PLOT Q4-3: Box plot: worldwide gross distribution (top 8 genres) 
top8_genres = q4_summary.head(8)['genre'].tolist()
q4_top8     = q4[q4['genre'].isin(top8_genres) & q4['worldwide'].notna()]

data_by_genre = [
    q4_top8[q4_top8['genre'] == g]['worldwide'].dropna().values / 1e6
    for g in top8_genres
]

fig, ax = plt.subplots(figsize=(13, 6))
bp = ax.boxplot(
    data_by_genre, labels=top8_genres, patch_artist=True,
    medianprops={'color': 'white', 'linewidth': 2.5},
    whiskerprops={'color': MUTED_COL, 'linewidth': 1.2},
    capprops={'color': MUTED_COL, 'linewidth': 1.2},
    flierprops={'marker': 'o', 'markersize': 2.5, 'alpha': 0.3,
                'markerfacecolor': MUTED_COL, 'markeredgecolor': 'none'},
)
for patch, color in zip(bp['boxes'], PALETTE):
    patch.set_facecolor(color); patch.set_alpha(0.55)

ax.set_ylabel('Worldwide Gross (USD Millions)')
ax.set_title('Q4 — Distribution of Worldwide Gross Revenue by Genre (Top 8)\n'
             '(White line = median  |  Box = IQR  |  Dots = outliers)')
ax.yaxis.set_major_formatter(mticker.FuncFormatter(lambda x, _: f'${x:.0f}M'))
ax.set_axisbelow(True)
ax.spines[['top','right']].set_visible(False)
plt.xticks(rotation=20, ha='right')
fig.tight_layout()
plt.show()

Action Epic and Superhero films show wide IQRs and dense high-value outliers, confirming that while most films earn hundreds of millions, a few cross one billion dollars. Computer Animation shows a more compact spread, indicating more consistent earnings across films in that genre.

In [ ]:
# Figure Q4-4: Mean vs median worldwide gross by genre (top 12). When the mean bar is substantially taller
# than the median bar, the genre's average is being pulled upward by a small number of very high-earning
# blockbusters. Genres where mean ≈ median have more evenly distributed revenue.

# PLOT Q4-4: Mean vs Median: revealing skewness per genre 
top12 = q4_summary.head(12)
x, w  = np.arange(len(top12)), 0.38

fig, ax = plt.subplots(figsize=(13, 6))
ax.bar(x - w/2, top12['avg_worldwide']    / 1e6, w,
       label='Mean',   color=CYAN, alpha=0.85, edgecolor='none')
ax.bar(x + w/2, top12['median_worldwide'] / 1e6, w,
       label='Median', color=GOLD, alpha=0.85, edgecolor='none')

ax.set_xticks(x); ax.set_xticklabels(top12['genre'], rotation=30, ha='right')
ax.set_ylabel('Worldwide Gross (USD Millions)')
ax.set_title('Q4 — Mean vs Median Worldwide Gross by Genre\n'
             '(large gap = distribution skewed by a small number of blockbusters)')
ax.yaxis.set_major_formatter(mticker.FuncFormatter(lambda x, _: f'${x:.0f}M'))
ax.legend(); ax.set_axisbelow(True)
ax.spines[['top','right']].set_visible(False)
fig.tight_layout()
plt.show()

# Statistical test: Kruskal-Wallis
groups = [
    q4[q4['genre'] == g]['worldwide'].dropna().values
    for g in valid_genres
    if q4[q4['genre'] == g]['worldwide'].notna().sum() >= 5
]
h_stat, p_val = stats.kruskal(*groups)
print(f"\nKruskal-Wallis test (worldwide gross across genres):")
print(f"  H = {h_stat:.2f},  p = {p_val:.2e}  {significance_stars(p_val)}")
print(f"  Interpretation: Genre groups differ significantly in worldwide revenue.")

Sword & Sorcery shows the most extreme gap between mean ($226M) and median ($9M), meaning a few franchise blockbusters inflate the group average far beyond what a typical film earns. The Kruskal-Wallis test (H = 6,126, p < 0.001) confirms these revenue differences across genres are statistically significant and not due to chance.

**Conclusion:** Genre is a strong and statistically significant predictor of box office revenue (KW H = 6,126, p < 0.001). Action-oriented and franchise genres consistently dominate, with most top genres earning more internationally than domestically. Group averages are heavily skewed by blockbuster outliers, making the median a more reliable measure of typical commercial performance.

## Q5: Is there a relationship between movie budget and box office performance?

This section examines whether spending more on a movie's production translates into higher box office earnings. It also investigates whether high-budget films are more profitable (in terms of return on investment) than low-budget ones, and whether budget size affects how front-loaded a movie's revenue is. These findings are relevant to anyone deciding how much to invest in a production.

### Data Cleaning

This step converts all budget and revenue columns from formatted text strings — which may appear in different currencies and include labels like "(estimated)" — into plain numeric USD values. Only rows where both the budget and the worldwide gross are available are kept, since both are needed to study their relationship. A ROI column is computed and used to detect and remove data entry errors: values below −100% (mathematically impossible) or above 10,000% (almost certainly a corrupted record) are excluded.

In [ ]:
# =============================================================================
# Q5 — DATA CLEANING
# Target columns: budget, opening_weekend_gross, gross_worldwide, gross_us_canada
# =============================================================================

q5_raw = df_raw[['id', 'title', 'primary_genre' if 'primary_genre' in df_raw.columns else 'genres',
                  'release_date',
                  'budget',
                  'opening_weekend_gross',
                  'gross_worldwide',
                  'gross_us_canada']].copy()

if 'primary_genre' not in q5_raw.columns:
    q5_raw = q5_raw.rename(columns={'genres': 'genres_raw'})
    q5_raw['primary_genre'] = q5_raw['genres_raw'].apply(
        lambda v: parse_list_col(v)[0] if parse_list_col(v) else None
    )

print(f"Starting rows: {len(q5_raw):,}")
print(f"\nMissing values before cleaning:")
print(q5_raw[['budget','opening_weekend_gross','gross_worldwide','gross_us_canada']].isnull().sum().to_string())

# Step 1 — Convert all money columns to numeric USD
for col in ['budget', 'opening_weekend_gross', 'gross_worldwide', 'gross_us_canada']:
    q5_raw[col + '_num'] = q5_raw[col].apply(parse_money)

# Step 2 — Filter: keep only rows where BOTH budget AND worldwide gross are present
q5 = q5_raw[
    q5_raw['budget_num'].notna() &
    q5_raw['gross_worldwide_num'].notna() &
    (q5_raw['budget_num'] > 0) &
    (q5_raw['gross_worldwide_num'] > 0)
].copy()

print(f"\nRows after requiring both budget and worldwide gross: {len(q5):,}")

# Step 3 — Engineer ROI and remove extreme outliers (ROI = (worldwide - budget) / budget × 100)
q5['roi'] = (q5['gross_worldwide_num'] - q5['budget_num']) / q5['budget_num'] * 100

# Remove ROI values outside [-100%, 10000%]
# Values below -100% are impossible (you cannot lose more than you spent).
# Values above 10000% are almost certainly data entry errors (e.g. $1 budget).
before = len(q5)
q5 = q5[(q5['roi'] >= -100) & (q5['roi'] <= 10_000)].copy()
print(f"Rows removed as ROI outliers: {before - len(q5)}")
print(f"Final Q5 rows: {len(q5):,}")
print(f"\nMissing values after cleaning:")
print(q5[['budget_num','gross_worldwide_num','opening_weekend_gross_num']].isnull().sum().to_string())

The key cleaning challenge for Q5 is that both `budget` and `gross_worldwide` are stored as formatted text strings in multiple currencies. Each is parsed to a numeric USD value using the shared `parse_money()` function. Rows are dropped if either the budget or the worldwide gross is missing, because both are required to study their relationship. An ROI column is computed and used to detect impossible or implausible values: any ROI below −100% (mathematically impossible) or above 10,000% (almost certainly a data entry error, typically caused by a near-zero budget value) is removed. This step retains all genuine high-ROI films like low-budget breakout hits while eliminating corrupted records.

### Data Transformation: Creating X Input and Y Output Variables

The budget column (continuous, in USD) is the main X input variable. Two engineered variables are added: **ROI** (return on investment), which measures how profitable a movie was relative to its production cost, and **opening ratio**, which measures what share of the total worldwide gross was earned on opening weekend alone. A third engineered variable, **budget tier**, groups movies into Low, Mid, and High categories to enable simple grouped comparisons alongside the continuous analysis.

In [ ]:
# =============================================================================
# Q5 — DATA TRANSFORMATION
# =============================================================================

# Select and rename analysis columns
q5 = q5[[
    'id', 'title', 'primary_genre', 'release_date',
    'budget_num', 'opening_weekend_gross_num',
    'gross_worldwide_num', 'gross_us_canada_num',
    'roi'
]].copy()

q5.columns = [
    'id', 'title', 'genre', 'release_date',
    'budget', 'opening_weekend', 'worldwide', 'us_canada',
    'roi'
]

# ENGINEERED VARIABLE 1: Opening Ratio = opening weekend gross / total worldwide gross
q5['opening_ratio'] = q5['opening_weekend'] / q5['worldwide']

# ENGINEERED VARIABLE 2: Budget Tier
def assign_tier(b):
    if pd.isna(b):       return None
    if b < 10_000_000:   return 'Low (<$10M)'
    if b <= 100_000_000: return 'Mid ($10M–$100M)'
    return 'High (>$100M)'

q5['budget_tier'] = q5['budget'].apply(assign_tier)

TIER_ORDER  = ['Low (<$10M)', 'Mid ($10M\u2013$100M)', 'High (>$100M)']
TIER_COLORS = {'Low (<$10M)': CORAL, 'Mid ($10M\u2013$100M)': CYAN, 'High (>$100M)': GOLD}

# SUMMARY TABLE by budget tier
q5_tier = (
    q5.groupby('budget_tier', observed=True)
    .agg(
        n_movies          = ('title',          'count'),
        avg_budget        = ('budget',          'mean'),
        avg_worldwide     = ('worldwide',        'mean'),
        median_worldwide  = ('worldwide',        'median'),
        avg_roi           = ('roi',              'mean'),
        median_roi        = ('roi',              'median'),
        pct_profitable    = ('roi', lambda x: (x > 0).mean() * 100),
        avg_opening_ratio = ('opening_ratio',   'mean'),
    )
    .reindex(TIER_ORDER)
    .reset_index()
    .rename(columns={'budget_tier': 'Budget Tier'})
)

disp5 = q5_tier.copy()
for col in ['avg_budget','avg_worldwide','median_worldwide']:
    disp5[col] = disp5[col].apply(lambda x: f'${x/1e6:.1f}M' if pd.notna(x) else 'N/A')
disp5['avg_roi']         = disp5['avg_roi'].apply(lambda x: f'{x:.0f}%')
disp5['median_roi']      = disp5['median_roi'].apply(lambda x: f'{x:.0f}%')
disp5['pct_profitable']  = disp5['pct_profitable'].apply(lambda x: f'{x:.1f}%')
disp5['avg_opening_ratio']= disp5['avg_opening_ratio'].apply(lambda x: f'{x:.2f}' if pd.notna(x) else 'N/A')

print("Summary Table — Performance by Budget Tier:\n")
print(disp5.to_string(index=False))

# Pearson correlation on log scale
log_b = np.log10(q5['budget'])
log_w = np.log10(q5['worldwide'])
r_bw, p_bw = stats.pearsonr(log_b, log_w)
print(f"\nPearson r (log budget vs log worldwide): r = {r_bw:.3f},  p = {p_bw:.2e}  {significance_stars(p_bw)}")

In [ ]:
# =============================================================================
# EXPORT Q5 FINAL DATASET
# =============================================================================
q5.to_csv("outputs/q5_budget_performance_dataset.csv", index=False)
q5_tier.to_csv("outputs/q5_budget_tier_summary.csv", index=False)

The transformation step produces two new engineered variables. **Opening Ratio** (opening weekend ÷ worldwide gross) is a continuous measure of how front-loaded a movie's revenue is — a ratio close to 1 means nearly all money was earned on launch day, while a low ratio indicates a slow-building film. **Budget Tier** converts the continuous budget variable into three ordered categories (Low, Mid, High) which allow grouped bar and violin comparisons. The grouping variable (X input) is `budget_tier` for categorical analysis and `budget` (continuous, log-scaled) for correlation analysis. The output variables (Y) are `worldwide`, `roi`, and `opening_ratio`.

### Data visualization

In [ ]:
# Figure Q5-1: Budget vs worldwide gross on a log-log scale, coloured by budget tier.
# The dashed white line is a linear regression fitted to the log-transformed values.
# The Pearson r on the log scale measures the strength of the monotonic relationship between budget and revenue.

# ── Apply budget cap to remove currency-parsing errors (78 rows, 0.8%) ──
q5 = q5[q5['budget'] <= 500_000_000].copy()

# Recompute log values on the cleaned data
log_b = np.log10(q5['budget'])
log_w = np.log10(q5['worldwide'])
r_bw, p_bw = stats.pearsonr(log_b, log_w)

# PLOT Q5-1: Scatter: Budget vs Worldwide Gross (log–log)
fig, ax = plt.subplots(figsize=(12, 7))
for tier in TIER_ORDER:
    sub = q5[q5['budget_tier'] == tier]
    ax.scatter(sub['budget'] / 1e6, sub['worldwide'] / 1e6,
               c=TIER_COLORS[tier], alpha=0.3, s=15,
               edgecolors='none', label=tier)

# Regression line on log-log data
x_range = np.linspace(log_b.min(), log_b.max(), 300)
slope, intercept, *_ = stats.linregress(log_b, log_w)
ax.plot(10**x_range / 1e6,
        10**(slope * x_range + intercept) / 1e6,
        color='white', lw=2, ls='--',
        label=f'Trend line  (r = {r_bw:.2f})')

ax.set_xscale('log'); ax.set_yscale('log')
ax.set_xlabel('Production Budget (USD Millions, log scale)')
ax.set_ylabel('Worldwide Gross (USD Millions, log scale)')
ax.set_title(f'Q5 — Production Budget vs Worldwide Gross Revenue\n'
             f'(Pearson r = {r_bw:.2f} on log scale, p {significance_stars(p_bw)})')
ax.xaxis.set_major_formatter(mticker.FuncFormatter(lambda x, _: f'${x:.4g}M'))
ax.yaxis.set_major_formatter(mticker.FuncFormatter(lambda x, _: f'${x:.4g}M'))
ax.legend(); ax.set_axisbelow(True)
ax.spines[['top','right']].set_visible(False)
fig.tight_layout()
plt.show()

The OLS trend line confirms a strong positive relationship between budget and worldwide gross (Pearson r = 0.70, p < 0.001) — higher-budget films tend to earn more. However, the wide scatter within each tier shows that budget alone does not guarantee success: some low-budget films far exceed the trend while several high-budget films fall below break-even.

In [ ]:
# Figure Q5-2: Average ROI by budget tier. Bars show the mean ROI; individual movie ROIs are overlaid
# as semi-transparent dots (jittered horizontally for visibility). The percentage below each bar label
# shows what share of movies in that tier turned a profit. The dashed white line marks the break-even
# point (ROI = 0).

# PLOT Q5-2: ROI by budget tier (bar + jitter) 
fig, ax = plt.subplots(figsize=(9, 6))
x_pos = np.arange(len(TIER_ORDER))

bars = ax.bar(
    x_pos,
    [q5_tier.loc[q5_tier['Budget Tier'] == t, 'avg_roi'].values[0] for t in TIER_ORDER],
    color=[TIER_COLORS[t] for t in TIER_ORDER],
    edgecolor='none', width=0.5, alpha=0.85, zorder=2
)

np.random.seed(42)
for i, tier in enumerate(TIER_ORDER):
    sub_roi = q5[q5['budget_tier'] == tier]['roi'].dropna()
    jitter  = np.random.uniform(-0.18, 0.18, size=len(sub_roi))
    ax.scatter(i + jitter, sub_roi, color=TIER_COLORS[tier],
               alpha=0.07, s=6, edgecolors='none', zorder=1)

ax.axhline(0, color='white', lw=1, ls='--', alpha=0.5)
ax.set_xticks(x_pos); ax.set_xticklabels(TIER_ORDER)
ax.set_ylabel('Return on Investment (%)')
ax.set_title('Q5 — Average ROI by Budget Tier\n'
             '(bars = mean ROI  |  dots = individual movies)')

for bar, tier in zip(bars, TIER_ORDER):
    h   = bar.get_height()
    n   = int(q5_tier.loc[q5_tier['Budget Tier'] == tier, 'n_movies'])
    pct = float(q5_tier.loc[q5_tier['Budget Tier'] == tier, 'pct_profitable'])
    off = 12 if h >= 0 else -22
    ax.text(bar.get_x() + bar.get_width() / 2, h + off,
            f'{h:.0f}%\nn={n}\n{pct:.0f}% profitable',
            ha='center', va='bottom', fontsize=8.5, color=TEXT_COL)

ax.set_axisbelow(True)
ax.spines[['top','right']].set_visible(False)
fig.tight_layout()
plt.show()

Low-budget films show the highest average ROI (209%) but only 39% are profitable, as the mean is driven by extreme outliers — the median ROI is actually −52%. High-budget films offer the most reliable returns with 65% profitability and a median ROI of 91%, reflecting better risk management through franchises and global marketing.

In [ ]:
# Figure Q5-3: Opening weekend revenue share (opening ratio) by budget tier, shown as violin plots.
# The width of the violin at each point represents the density of movies at that ratio value.
# The white horizontal line is the median. High-budget movies tend to have higher opening ratios,
# reflecting the blockbuster launch strategy.

# PLOT Q5-3: Opening ratio by budget tier 
q5_ratio = q5[
    q5['opening_ratio'].notna() &
    (q5['opening_ratio'] > 0) &
    (q5['opening_ratio'] <= 1)
]

violin_data = [q5_ratio[q5_ratio['budget_tier'] == t]['opening_ratio'].values
               for t in TIER_ORDER]

fig, ax = plt.subplots(figsize=(9, 6))
parts = ax.violinplot(violin_data, positions=[1, 2, 3],
                      showmedians=True, showextrema=False)
for body, tier in zip(parts['bodies'], TIER_ORDER):
    body.set_facecolor(TIER_COLORS[tier]); body.set_alpha(0.55)
parts['cmedians'].set_color('white'); parts['cmedians'].set_linewidth(2.5)

for i, tier in enumerate(TIER_ORDER, start=1):
    med = q5_ratio[q5_ratio['budget_tier'] == tier]['opening_ratio'].median()
    ax.text(i, med + 0.02, f'{med:.2f}',
            ha='center', va='bottom', fontsize=9, color='white')

ax.set_xticks([1, 2, 3]); ax.set_xticklabels(TIER_ORDER)
ax.set_ylabel('Opening Weekend / Total Worldwide Gross')
ax.set_title('Q5 — Opening Weekend Revenue Share by Budget Tier\n'
             '(higher = more front-loaded; blockbuster launch pattern)')
ax.set_axisbelow(True)
ax.spines[['top','right']].set_visible(False)
fig.tight_layout()
plt.show()

High-budget films have the lowest median opening ratio (0.11), consistent with the blockbuster strategy of sustaining long theatrical runs through heavy marketing investment. Low-budget films show the widest spread, with many limited releases earning nearly all revenue on opening weekend before rapidly exiting cinemas.

In [ ]:
# Figure Q5-4: Median worldwide gross heatmap broken down by genre and budget tier.
# Each cell shows the median worldwide gross (in USD millions) for movies in that genre–tier combination.
# Empty cells indicate fewer than 5 movies in that combination.
# This reveals which genre–budget combinations are most commercially successful.

# PLOT Q5-4: Heatmap — Median worldwide gross by genre × budget tier
pivot = (
    q5[q5['genre'].notna()]
    .groupby(['genre', 'budget_tier'])['worldwide']
    .median()
    .unstack('budget_tier')
    .reindex(columns=TIER_ORDER)
)
pivot = pivot[pivot.notna().sum(axis=1) >= 2].dropna(thresh=2)
pivot = pivot.sort_values('High (>$100M)', ascending=False).head(14)
pivot_display = pivot / 1e6

fig, ax = plt.subplots(figsize=(10, 8))
sns.heatmap(
    pivot_display, ax=ax, annot=True, fmt='.0f', cmap='YlOrRd',
    linewidths=0.5, linecolor=DARK_BG,
    cbar_kws={'label': 'Median Worldwide Gross (USD Millions)', 'shrink': 0.8},
    annot_kws={'size': 9}
)
ax.set_title('Q5 — Median Worldwide Gross (USD M) by Genre \u00d7 Budget Tier\n'
             '(empty = fewer than 5 movies in that combination)')
ax.set_xlabel('Budget Tier'); ax.set_ylabel('Genre')
ax.tick_params(axis='x', rotation=0); ax.tick_params(axis='y', rotation=0)
fig.tight_layout()
plt.show()

The budget–revenue relationship is strongly genre-dependent: Superhero and Computer Animation films reach median worldwide grosses of ~$500M at high budgets, while other genres show far smaller gains. Genres like Dinosaur Adventure show a steep step-up across tiers ($0M → $85M → $525M), suggesting audiences require a minimum level of spectacle that only high budgets can deliver.

**Conclusion:** Production budget and worldwide gross are positively correlated (r = 0.70, p < 0.001), confirming that higher-budget films tend to earn more. However, high-budget films are the most reliably profitable by median ROI (91%) and profitability rate (65%), while low-budget films carry the highest financial risk despite a misleadingly high average ROI. The commercial impact of budget also depends heavily on genre, making genre–budget pairing the most actionable insight for production investment decisions.

## Q6: Do movies with well-known directors, writers, or stars receive higher ratings or more votes?

This section investigates whether the prominence of a movie's talent; its cast, director, and writers, has a measurable effect on how audiences rate the film and how many people engage with it on IMDb. This is relevant for understanding whether star power drives audience perception, or whether other factors matter more.

### Data Cleaning

This step prepares the three talent columns (stars, directors, writers) and the two outcome columns (rating, votes) for analysis. The talent columns store lists as plain text strings, which are parsed back into real Python lists. The rating column is converted from text to a numeric float. The votes column uses shorthand notation such as "8.3K" or "1.2M", which is parsed into a plain integer. Rows where either the rating or the vote count is missing are dropped, as both are required as outcome variables.

In [ ]:
# =============================================================================
# Q6 — DATA CLEANING
# Target columns: stars, directors, writers, rating, votes
# =============================================================================

q6_raw = df_raw[['id', 'title', 'stars', 'directors', 'writers',
                  'rating', 'votes']].copy()

print(f"Starting rows: {len(q6_raw):,}")
print(f"\nMissing values before cleaning:")
print(q6_raw.isnull().sum().to_string())

# Step 1 — Parse stringified list columns into real Python lists
q6_raw['stars_list']     = q6_raw['stars'].apply(parse_list_col)
q6_raw['directors_list'] = q6_raw['directors'].apply(parse_list_col)
q6_raw['writers_list']   = q6_raw['writers'].apply(parse_list_col)

# Step 2 — Convert rating to numeric float
q6_raw['rating_num'] = pd.to_numeric(q6_raw['rating'], errors='coerce')

# Step 3 — Parse votes (e.g. '8.3K', '1.2M') to integer
q6_raw['votes_num'] = q6_raw['votes'].apply(parse_votes)

# Step 4 — Filter: both rating and votes must be present
q6_base = q6_raw[
    q6_raw['rating_num'].notna() &
    q6_raw['votes_num'].notna()
].copy()

print(f"\nRows after requiring both rating and votes: {len(q6_base):,}")
print(f"Rating range: {q6_base['rating_num'].min():.1f} – {q6_base['rating_num'].max():.1f}")
print(f"Votes range:  {q6_base['votes_num'].min():,} – {q6_base['votes_num'].max():,}")

The cleaning process for Q6 addresses three distinct challenges. First, the `stars`, `directors`, and `writers` columns store lists as plain text strings — these are parsed back into real Python lists using `ast.literal_eval`. Second, the `rating` column is read as text and converted to a float; invalid entries (e.g. empty strings) become `NaN` and are excluded. Third, the `votes` column uses shorthand notation like `"8.3K"` or `"1.2M"` which is parsed into a plain integer. Rows where either the rating or vote count is missing are dropped, because both are required as outcome variables for this question.

### Data Transformation: Creating X Input and Y Output Variables

Since the dataset contains no external fame or popularity measure, "well-known" is defined using an **appearance count** proxy: for each person, we count how many unique movies they appear in across the full dataset. The maximum score among all people listed in a given role is assigned to each movie (so one prominent person is enough). Movies are then split into **High-profile** (top 25% by appearance count) and **Low-profile** (bottom 75%) for each role separately. These profile tier labels are the X input variables; `rating_num` and `votes_num` are the Y output variables.

In [ ]:
# =============================================================================
# Q6 — DATA TRANSFORMATION
# =============================================================================

def compute_fame(list_col, role_label):
    exploded = (
        df_raw[['id', list_col]]
        .assign(**{list_col: df_raw[list_col].apply(parse_list_col)})
        .explode(list_col)
        .dropna(subset=[list_col])
        .rename(columns={list_col: 'person'})
    )
    exploded['person'] = exploded['person'].str.strip()
    exploded = exploded[exploded['person'] != '']

    fame_count = (
        exploded.groupby('person')['id']
        .nunique()
        .reset_index()
        .rename(columns={'id': f'{role_label}_fame'})
    )
    merged    = exploded.merge(fame_count, on='person', how='left')
    movie_max = merged.groupby('id')[f'{role_label}_fame'].max().reset_index()
    return movie_max

star_fame     = compute_fame('stars',     'star')
director_fame = compute_fame('directors', 'director')
writer_fame   = compute_fame('writers',   'writer')

# Merge all three fame scores onto q6_base
q6 = q6_base[['id','title','rating_num','votes_num',
               'stars_list','directors_list','writers_list']].copy()
q6 = q6.merge(star_fame,     on='id', how='left')
q6 = q6.merge(director_fame, on='id', how='left')
q6 = q6.merge(writer_fame,   on='id', how='left')

# Combined fame: average of the three role scores
q6['combined_fame'] = q6[['star_fame','director_fame','writer_fame']].mean(axis=1)

# Profile tiers (computed separately per role so each threshold is role-specific)
ROLES       = ['star',        'director',        'writer']
FAME_COLS   = ['star_fame',   'director_fame',   'writer_fame']
TIER_COLS   = ['star_profile_tier','director_profile_tier','writer_profile_tier']
ROLE_LABELS = {'star':'Stars','director':'Directors','writer':'Writers'}
ROLE_COLORS = {'star': GOLD,  'director': CYAN,      'writer': CORAL}

for role, fame_col, tier_col in zip(ROLES, FAME_COLS, TIER_COLS):
    threshold  = q6[fame_col].quantile(0.75)
    q6[tier_col] = q6[fame_col].apply(
        lambda x: 'High-profile' if pd.notna(x) and x >= threshold else 'Low-profile'
    )

# Credit count variables
q6['n_stars']     = q6['stars_list'].apply(len)
q6['n_directors'] = q6['directors_list'].apply(len)
q6['n_writers']   = q6['writers_list'].apply(len)

print("Transformation complete. Q6 columns:")
print([c for c in q6.columns if c not in ['stars_list','directors_list','writers_list']])

# Summary table
rows = []
for role, fame_col, tier_col in zip(ROLES, FAME_COLS, TIER_COLS):
    for tier in ['High-profile', 'Low-profile']:
        sub = q6[q6[tier_col] == tier]
        rows.append({
            'Role'      : ROLE_LABELS[role],
            'Profile'   : tier,
            'N'         : len(sub),
            'Avg Rating': f"{sub['rating_num'].mean():.2f}",
            'Med Rating': f"{sub['rating_num'].median():.2f}",
            'Avg Votes' : f"{sub['votes_num'].mean()/1e3:.1f}K",
            'Med Votes' : f"{sub['votes_num'].median()/1e3:.1f}K",
        })
print("\nSummary Table — High-profile vs Low-profile:\n")
print(pd.DataFrame(rows).to_string(index=False))

In [ ]:
# =============================================================================
# EXPORT Q6 FINAL DATASET
# =============================================================================

q6.to_csv("outputs/q6_talent_impact_dataset.csv", index=False)

The central engineering decision for Q6 is how to operationalize "well-known" without an external fame database. The solution uses **appearance count** — the number of unique movies each person appears in across the full dataset — as a proxy for industry prominence. For each movie the maximum fame score among all listed people in a given role is used, so that one highly prominent person is sufficient to classify a movie as high-profile. The 75th percentile of the fame score distribution within each role is used as the threshold separating High-profile (top 25%) from Low-profile (bottom 75%) movies. The grouping tier is the X input variable; `rating_num` and `votes_num` are the two Y output variables.

### Data visualization

In [ ]:
# Figure Q6-1: Average IMDb rating for high-profile vs low-profile talent, shown separately for stars,
# directors, and writers. Error bars represent 95% confidence intervals. The Δ value above each pair
# of bars shows the rating difference between the two groups.

# PLOT Q6-1: Bar chart — Mean rating: High vs Low profile per role
fig, axes = plt.subplots(1, 3, figsize=(15, 6), sharey=True)
fig.suptitle('Q6 — Average IMDb Rating: High-Profile vs Low-Profile Talent',
             fontsize=14, fontweight='bold', y=1.01)

for ax, role, tier_col in zip(axes, ROLES, TIER_COLS):
    color = ROLE_COLORS[role]
    means, errs = {}, {}
    for tier in ['High-profile', 'Low-profile']:
        sub = q6[q6[tier_col] == tier]['rating_num'].dropna()
        means[tier] = sub.mean()
        errs[tier]  = sub.sem() * 1.96

    bars = ax.bar(['Low-profile', 'High-profile'],
                  [means['Low-profile'], means['High-profile']],
                  yerr=[errs['Low-profile'], errs['High-profile']],
                  color=['#3a3a5c', color], capsize=5,
                  edgecolor='none', width=0.5,
                  error_kw={'ecolor': 'white', 'alpha': 0.7})

    diff = means['High-profile'] - means['Low-profile']
    ax.set_title(f'{ROLE_LABELS[role]}\n(\u0394 = {diff:+.2f} rating points)')
    ax.set_ylabel('Average IMDb Rating' if role == 'star' else '')
    ax.set_ylim(0, 10); ax.set_axisbelow(True)
    ax.spines[['top','right']].set_visible(False)
    for bar in bars:
        h = bar.get_height()
        ax.text(bar.get_x() + bar.get_width() / 2, h + 0.1,
                f'{h:.2f}', ha='center', va='bottom', fontsize=10, color=TEXT_COL)

fig.tight_layout()
plt.show()

Rating differences between High-profile and Low-profile talent are negligible across all three roles (Δ ≤ +0.08 points), despite large sample sizes that estimate these averages precisely. Well-known talent does not meaningfully predict a higher-rated film as measured by IMDb scores.

In [ ]:
# Figure Q6-2: Average number of IMDb votes for high-profile vs low-profile talent, separately for each role.
# Votes are shown in thousands. The Δ value shows the difference in average vote count between
# the two profile tiers.

# PLOT Q6-2: Bar chart: Mean votes: High vs Low profile per role 
fig, axes = plt.subplots(1, 3, figsize=(15, 6))
fig.suptitle('Q6 — Average Number of Votes: High-Profile vs Low-Profile Talent',
             fontsize=14, fontweight='bold', y=1.01)

for ax, role, tier_col in zip(axes, ROLES, TIER_COLS):
    color = ROLE_COLORS[role]
    means, errs = {}, {}
    for tier in ['High-profile', 'Low-profile']:
        sub = q6[q6[tier_col] == tier]['votes_num'].dropna()
        means[tier] = sub.mean()
        errs[tier]  = sub.sem() * 1.96

    ax.bar(['Low-profile', 'High-profile'],
           [means['Low-profile'] / 1e3, means['High-profile'] / 1e3],
           yerr=[errs['Low-profile'] / 1e3, errs['High-profile'] / 1e3],
           color=['#3a3a5c', color], capsize=5,
           edgecolor='none', width=0.5,
           error_kw={'ecolor': 'white', 'alpha': 0.7})

    diff = (means['High-profile'] - means['Low-profile']) / 1e3
    ax.set_title(f'{ROLE_LABELS[role]}\n(\u0394 = {diff:+.0f}K votes)')
    ax.set_ylabel('Average Votes (thousands)' if role == 'star' else '')
    ax.set_axisbelow(True)
    ax.spines[['top','right']].set_visible(False)

fig.tight_layout()
plt.show()

Counterintuitively, High-profile directors and writers are associated with far fewer votes (Δ = −13K and −14K respectively), because appearance count captures prolific industry workers rather than mainstream celebrity. Films attracting millions of votes are typically directed by filmmakers with only a few major productions, giving them a low appearance count in this dataset.

In [ ]:
# Figure Q6-3: Scatter plots of talent appearance count (fame proxy) vs IMDb rating,
# for stars, directors, and writers separately. Each dot is one movie.
# The white line is the decile mean trend, computed by dividing movies into 10 equal fame-score groups
# and plotting the group average rating. This reveals whether higher fame consistently associates
# with higher or lower ratings, without assuming a linear relationship.

# PLOT Q6-3: Scatter — Fame score vs Rating (decile trend overlay) 
fig, axes = plt.subplots(1, 3, figsize=(17, 6))
fig.suptitle('Q6 — Talent Appearance Count vs IMDb Rating\n'
             '(dots = individual movies  |  white line = decile mean trend)',
             fontsize=13, fontweight='bold', y=1.01)

for ax, role, fame_col in zip(axes, ROLES, FAME_COLS):
    color = ROLE_COLORS[role]
    sub   = q6[q6[fame_col].notna() & q6['rating_num'].notna()].copy()

    ax.scatter(sub[fame_col], sub['rating_num'],
               c=color, alpha=0.1, s=8, edgecolors='none')

    sub['decile'] = pd.qcut(sub[fame_col], 10, labels=False, duplicates='drop')
    binned = sub.groupby('decile').agg(
        mean_rating=('rating_num', 'mean'),
        mean_fame  =(fame_col,     'mean'),
    ).reset_index()
    ax.plot(binned['mean_fame'], binned['mean_rating'],
            color='white', lw=2.5, marker='o', markersize=5, zorder=3)

    r_s, p_s = stats.spearmanr(sub[fame_col], sub['rating_num'])
    ax.set_title(f'{ROLE_LABELS[role]}\n'
                 f'(Spearman r = {r_s:.2f},  p {significance_stars(p_s)})')
    ax.set_xlabel(f'{ROLE_LABELS[role][:-1]} Appearance Count')
    ax.set_ylabel('IMDb Rating' if role == 'star' else '')
    ax.set_axisbelow(True)
    ax.spines[['top','right']].set_visible(False)

fig.tight_layout()
plt.show()

Spearman correlations are near-zero across all roles (r = −0.03 for Stars, +0.03 for Directors and Writers) and the decile trend lines are nearly flat, confirming that fame score explains almost none of the variance in individual film ratings. The statistically significant p-values reflect the very large sample size rather than any meaningful relationship.

In [ ]:
# Figure Q6-4: Box plots comparing the full distribution of IMDb ratings between high-profile and
# low-profile talent groups, for each role. The white line is the median. The Mann-Whitney U test
# p-value and significance level are shown in the title of each panel. This test is used instead of
# a t-test because vote and rating distributions are heavily skewed.

# PLOT Q6-4: Box plots — Rating distribution, High vs Low
fig, axes = plt.subplots(1, 3, figsize=(15, 6), sharey=True)
fig.suptitle('Q6 — IMDb Rating Distribution: High-Profile vs Low-Profile Talent',
             fontsize=14, fontweight='bold', y=1.01)

for ax, role, tier_col in zip(axes, ROLES, TIER_COLS):
    color = ROLE_COLORS[role]
    high  = q6[q6[tier_col] == 'High-profile']['rating_num'].dropna().values
    low   = q6[q6[tier_col] == 'Low-profile' ]['rating_num'].dropna().values

    bp = ax.boxplot(
        [low, high], labels=['Low-profile', 'High-profile'],
        patch_artist=True,
        medianprops={'color': 'white', 'linewidth': 2.5},
        whiskerprops={'color': MUTED_COL, 'linewidth': 1.2},
        capprops={'color': MUTED_COL, 'linewidth': 1.2},
        flierprops={'marker': 'o', 'markersize': 2, 'alpha': 0.2,
                    'markerfacecolor': MUTED_COL, 'markeredgecolor': 'none'},
    )
    bp['boxes'][0].set_facecolor('#3a3a5c'); bp['boxes'][0].set_alpha(0.75)
    bp['boxes'][1].set_facecolor(color);    bp['boxes'][1].set_alpha(0.75)

    u, p = stats.mannwhitneyu(high, low, alternative='two-sided')
    ax.set_title(f'{ROLE_LABELS[role]}\n(MWU p = {p:.3f}  {significance_stars(p)})')
    ax.set_ylabel('IMDb Rating' if role == 'star' else '')
    ax.set_axisbelow(True)
    ax.spines[['top','right']].set_visible(False)

fig.tight_layout()
plt.show()

# Print full statistical test results
print("Statistical Test Results — Mann-Whitney U (rating):")
print(f"{'Role':<12} {'High median':>12} {'Low median':>12} {'p-value':>12} {'Sig':>5}")
print("-" * 58)
for role, tier_col in zip(ROLES, TIER_COLS):
    high = q6[q6[tier_col] == 'High-profile']['rating_num'].dropna()
    low  = q6[q6[tier_col] == 'Low-profile' ]['rating_num'].dropna()
    u, p = stats.mannwhitneyu(high, low, alternative='two-sided')
    print(f"{ROLE_LABELS[role]:<12} {high.median():>12.2f} {low.median():>12.2f} "
          f"{p:>12.2e} {significance_stars(p):>5}")

The median rating is identical at 6.30 across all six groups regardless of role or profile tier, and the IQRs overlap almost perfectly. Although the MWU test reaches significance for Directors (p = 0.015) and Writers (p = 0.009), the identical medians confirm these results carry no practical meaning; talent prominence has no meaningful effect on audience ratings.

**Conclusion:** Talent prominence, as measured by appearance count, has no meaningful effect on audience ratings, differences are below 0.08 points across all roles with identical medians of 6.30. Its counterintuitive negative association with vote counts reflects a limitation of the proxy rather than a real relationship. Overall, what is on screen matters far more than who is on screen, at least as reflected in IMDb scores.

# CLOSING THE WORK STATION, RESULTS TO BE DISCUSSED IN THE "MAIN WORK STATION FOR ALL.qmd"